# Correlation Mapping:

### Correlation Matrix:

In [10]:
import pandas as pd
import numpy as np

# Constants from metrics doc
MIN_TICKERS = 5

FILE_PATH = "../../data/processed/gdelt_ohlcv_join.parquet"

df = pd.read_parquet(FILE_PATH)

# Check data sparsity 
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Date range:", df["price_date"].min(), "->", df["price_date"].max())
print("Unique ticker-days:", df[["ticker", "price_date"]].drop_duplicates().shape[0])

# Parse dates
df["price_date"] = pd.to_datetime(df["price_date"], utc=True)
df["article_date"] = pd.to_datetime(df["article_date"], utc=True)

# Ensure numeric
num_cols = [
    "sentiment_score",
    "next_open",
    "next_close"
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Returns using metrics doc formula
# r_i,t = (close - open) / open

df["ret"] = (df["next_close"] - df["next_open"]) / df["next_open"]


# Aggregate sentiment to daily level (t)

daily_sentiment = (
    df.groupby("price_date")
      .agg(
          sentiment_score=("sentiment_score", "mean")
      )
      .reset_index()
)

# Compute cross-sectional metrics per day

returns = (
    df.groupby(["ticker", "price_date"])
      .agg(ret=("ret", "first"))
      .reset_index()
)

cs_stats = (
    returns.groupby("price_date")
    .agg(
        ret_cs_std=("ret", "std"),
        ret_mean=("ret", "mean"),
        n_tickers_returns=("ticker", "nunique"),
        ret_cs_mad=("ret", lambda x: np.mean(np.abs(x - np.mean(x))))
    )
    .reset_index()
)

# coverage ratio (7 tickers total)
cs_stats["coverage_ratio"] = cs_stats["n_tickers_returns"] / 7

# ticker coverage threshold
cs_stats = cs_stats[cs_stats["n_tickers_returns"] >= MIN_TICKERS]


# Merge sentiment with dispersion metrics

corr_df = daily_sentiment.merge(cs_stats, on="price_date")

corr_df = corr_df.dropna()


# Correlation matrices

cols = [
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean"
]

print("\nPearson Correlation Matrix\n")
print(corr_df[cols].corr(method="pearson"))

print("\nSpearman Correlation Matrix\n")
print(corr_df[cols].corr(method="spearman"))

Rows: 12523
Columns: ['seendate', 'url', 'title', 'language', 'domain', 'socialimage', 'company', 'ticker', 'date', 'sentiment_score', 'sentiment_hits', 'sentiment_present', 'article_date', 'price_date', 'next_open', 'next_high', 'next_low', 'next_close', 'next_adj_close', 'next_volume']
Date range: 2024-02-09 00:00:00 -> 2026-02-24 00:00:00
Unique ticker-days: 361

Pearson Correlation Matrix

                 sentiment_score  ret_cs_std  ret_cs_mad  ret_mean
sentiment_score         1.000000   -0.262100   -0.265465  0.136661
ret_cs_std             -0.262100    1.000000    0.969686 -0.127089
ret_cs_mad             -0.265465    0.969686    1.000000 -0.071656
ret_mean                0.136661   -0.127089   -0.071656  1.000000

Spearman Correlation Matrix

                 sentiment_score  ret_cs_std  ret_cs_mad  ret_mean
sentiment_score         1.000000   -0.113460   -0.123326 -0.006342
ret_cs_std             -0.113460    1.000000    0.975617  0.143904
ret_cs_mad             -0.123326    0

### Rolling Correlations:

In [ ]:
corr_df = corr_df.sort_values("price_date")

windows = [5, 10, 30]

for w in windows:
    
    corr_df[f"roll_corr_sent_std_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_cs_std"])
    )

    corr_df[f"roll_corr_sent_mad_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_cs_mad"])
    )

    corr_df[f"roll_corr_sent_mean_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_mean"])
    )

print("\nRolling correlation preview:\n")
print(
    corr_df[
        [
            "price_date",
            "roll_corr_sent_std_5d",
            "roll_corr_sent_std_10d",
            "roll_corr_sent_std_30d",
        ]
    ].tail(10)
)

# Diagnostics for sparsity and rolling window coverage

print("\nNon-null rolling counts:")
print(corr_df.filter(like="roll_corr").count())

print("\nFinal daily rows after MIN_TICKERS filter:", len(corr_df))
print("\nFirst available dates:")
print(corr_df["price_date"].sort_values().head(10))


Rolling correlation preview:

                  price_date  roll_corr_sent_std_5d  roll_corr_sent_std_10d  \
34 2026-02-06 00:00:00+00:00               0.348241               -0.591303   
35 2026-02-09 00:00:00+00:00              -0.153920               -0.616123   
36 2026-02-10 00:00:00+00:00              -0.924644               -0.650168   
37 2026-02-11 00:00:00+00:00              -0.945713               -0.637342   
38 2026-02-12 00:00:00+00:00              -0.954247               -0.439097   
39 2026-02-17 00:00:00+00:00              -0.677186               -0.566295   
40 2026-02-18 00:00:00+00:00               0.587980               -0.340972   
41 2026-02-19 00:00:00+00:00               0.360791               -0.570527   
42 2026-02-23 00:00:00+00:00               0.534026               -0.462315   
43 2026-02-24 00:00:00+00:00               0.760498               -0.217005   

    roll_corr_sent_std_30d  
34               -0.404668  
35               -0.403652  
36          

### Export Summary Tables 

In [12]:
summary_cols = [
    "price_date",
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean",
    "n_tickers_returns",
    "coverage_ratio",
    "roll_corr_sent_std_5d",
    "roll_corr_sent_std_10d",
    "roll_corr_sent_std_30d",
    "roll_corr_sent_mad_5d",
    "roll_corr_sent_mad_10d",
    "roll_corr_sent_mad_30d",
    "roll_corr_sent_mean_5d",
    "roll_corr_sent_mean_10d",
    "roll_corr_sent_mean_30d",
]

summary_df = corr_df[summary_cols].copy()

print("\nRows exported:", len(summary_df))
print("Date range:", summary_df["price_date"].min(), "->", summary_df["price_date"].max())

output_path = "correlation_summary.csv"
summary_df.to_csv(output_path, index=False)

print(f"Saved summary table to {output_path}")
print(summary_df.head())


Rows exported: 44
Date range: 2024-02-09 00:00:00+00:00 -> 2026-02-24 00:00:00+00:00
Saved summary table to correlation_summary.csv
                 price_date  sentiment_score  ret_cs_std  ret_cs_mad  \
0 2024-02-09 00:00:00+00:00         0.135330    0.011921    0.009208   
1 2024-04-09 00:00:00+00:00         0.227778    0.014713    0.010803   
2 2024-07-08 00:00:00+00:00         0.213771    0.013530    0.008924   
3 2024-07-09 00:00:00+00:00         0.139840    0.019864    0.013029   
4 2024-09-09 00:00:00+00:00         0.101538    0.012120    0.007676   

   ret_mean  n_tickers_returns  coverage_ratio  roll_corr_sent_std_5d  \
0  0.011527                  7             1.0                    NaN   
1 -0.001553                  7             1.0                    NaN   
2 -0.000446                  7             1.0                    NaN   
3  0.003720                  7             1.0                    NaN   
4 -0.001497                  7             1.0               0.082473